# 46 · Does the central claim survive uncertainty in the segment labels? (cross-fitted)

Referee issue M1: the human S1/S2/S3 labels are weakly supported. Notebook 42 transferred labels
from external S1/S2/S3 references by expression alone, and human agreement with the reviewed
labels was 0.71 (κ 0.54); 65% of human "S2" transferred to Lake S3. Here we ask whether notebook
43's central result, a low conservation index (cross-species agreement of gene zonation relative
to same-species ceilings), survives other labels.

**Circularity guard.** Genes that drive a label transfer will look zonated under the transferred
labels. Genes are split into two fixed sha256 halves. Labels are transferred with the
reference-zonated genes of one half and evaluated only on the other half; then the halves swap.

| Setting | Human labels | Mouse labels |
|---|---|---|
| (i) | Lake/KPMP transfer (centred) | reviewed |
| (ii) | Lake/KPMP transfer (centred) | microdissection transfer (centred) |
| (iii) | reviewed | reviewed (baseline: the effect of halving genes alone) |

Notebook 43 cannot restrict its evaluation genes, so this notebook re-implements only its primary
conservation index and its symmetric class counts from the same shared modules. A guard checks
that it reproduces notebook 43 exactly on the reviewed labels with all genes.

**Decision rule** (workstream-1 plan, fixed before any result): the claim "largely
species-specific" stands if, in every setting and both evaluation halves, the primary index is
≤ 0.5 with a 95% upper bound < 0.6 for both S2 − S1 and S3 − early. Species-only counts are
"unskewed" if mouse-only ÷ human-only < 2 under the symmetric setting in S2 − S1 and S3c − early.
Two mice and two cortex sections from one human donor; descriptive.

## 1 · Folders and settings (notebook 43's constants)

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
reference43 = results_root / 'pt_revision_classes' / 'reviewed' / 'tables'
external = data_root / 'external'
output = results_root / 'pt_revision_classes' / 'crossfit'
tables = output / 'tables'
tables.mkdir(parents=True, exist_ok=True)
label_dir = results_root / 'pt_revision_labels' / 'label_sets'
label_dir.mkdir(parents=True, exist_ok=True)
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
probe_file = results_root / 'pt_literature_deep' / 'probe_counts.csv'
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad',
            reference43 / 'conservation_index.csv', reference43 / 'class_counts_by_setting_and_contrast.csv',
            results_root / 'pt_literature_deep' / 'lake_cortex_segment_means.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz', lake_file, probe_file]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '46.crossfit_labels.1'
SEED, N_BOOT = 43, 1000                                   # notebook 43
MIN_COUNTS, MIN_NUCLEI, REFERENCE_LEVEL = 20, 50, 3.      # notebook 43
ALPHA, EFFECT, MARGIN, CONFIRM_ALPHA = .05, .5, .3, .05   # notebook 43
MIN_DETECTION, MIN_REFERENCE_LEVEL, MIN_REFERENCE_DIFFERENCE = .10, 1., 1.   # notebook 42 transfer
INDEX_MAX, INDEX_UPPER_MAX, SKEW_RATIO = .5, .6, 2.
print('Results folder:', output)

## 2 · Inputs (as notebook 43, sections 2–4)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial import cKDTree
from IPython.display import display

import pseudospace.zonation_amplitude as amplitude_module
import pseudospace.zonation_classes as classes_module
import pseudospace.zonation_reliability as reliability_module
from pseudospace.anchor_coordinate import bulk_segment_profiles
from pseudospace.coordinate_evaluation import sha256_folds
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.repeated_inputs import _read_and_validate_segmentation
from pseudospace.stage_cache import cached_payload, digest
from pseudospace.zonation_amplitude import group_sums, log_ratio
from pseudospace.zonation_classes import confirm, cross_species_class, moderated_contrast, union_class, zonation_calls
from pseudospace.zonation_reliability import confirm_flat, conservation_index, paired_gradient

cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(amplitude_module.__file__), Path(classes_module.__file__),
               Path(reliability_module.__file__)])
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in names}
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
measured = adata.var.measured_in_both_inputs.to_numpy()
gene_space = adata.var_names[measured]
counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
lognorm = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)[:, measured]
specimen = adata.obs['sample'].astype(str).to_numpy()
species = adata.obs.comparison_species.astype(str).to_numpy()
REVIEWED = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
library = np.asarray(counts.sum(axis=1)).ravel()
HALF = pd.Series(sha256_folds(gene_space, 2), index=gene_space)
print(f'{len(gene_space):,} genes; halves {HALF.value_counts().to_dict()}')


def mouse_depth():
    pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
    anchors = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
    pass1.file.close()
    values = np.full(len(specimen), np.nan)
    for name in expected['mouse']:
        native = ad.read_h5ad(source_paths[name], backed='r')
        source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
        native.file.close()
        mask = specimen == name
        structures = source_obs.loc[adata.obs_names[mask].str[len(name) + 1:]]
        _read_and_validate_segmentation(data_root, name, structures)
        glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
        _read_and_validate_segmentation(data_root, name, glomeruli)
        distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
            structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
        values[mask] = distance.mean(axis=1)
    return {'depth': values}


depth = cached_payload('mouse_depth', mouse_depth, root=cache, inputs={'obs': list(adata.obs_names)}, code=code)['depth']

# External references used by the primary index and the symmetric setting: mouse snRNA (male,
# female) and Lake/KPMP healthy cortex. Code as notebook 43, section 4.
CONTRASTS = {'S2-S1': ('S2', 'S1'), 'S3c-early': ('S3c', 'early'), 'S3-early': ('S3', 'early')}


def census_segments(label, segment_map):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_NUCLEI)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    counts_out, totals, info = {}, {}, {}
    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        for seg_name in needed:
            index = block.index[block.segment.eq(seg_name)]
            counts_out[(donor, seg_name)] = table.loc[index].sum()
            totals[(donor, seg_name)] = meta.loc[index, 'total_counts_all_genes'].sum()
        info[donor] = sex
    return pd.DataFrame(counts_out).reindex(gene_space).fillna(0), pd.Series(totals), pd.Series(info)


sn_counts, sn_totals, sn_sex = census_segments('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                          'epithelial cell of proximal tubule segment 2': 'S2',
                                                          'epithelial cell of proximal tubule segment 3': 'S3'})


def lake_pseudobulk():
    """Lake/KPMP cortex PT nuclei summed per donor x disease x segment (notebook 43's function)."""
    import h5py
    from pseudospace.replication_rules import csr_group_sums

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        cortex = obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
        keep = cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3'])
        label = (obs.donor_id + '|' + obs.disease + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
    human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T.reindex(columns=gene_space, fill_value=0.)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': frame.to_numpy()}


lake = cached_payload('lake_pseudobulk', lake_pseudobulk, root=cache,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'all'},
                      inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'genes': list(gene_space),
                              'orthologs': digest(orthologs)}, code=code)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'disease', 'sex', 'segment'])
lake_meta['nuclei'], lake_meta['total'] = lake['nuclei'], lake['totals']
lake_sums = pd.DataFrame(lake['sums'], columns=gene_space)
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_NUCLEI).all(axis=1)]
lake_disease = lake_meta.drop_duplicates('donor').set_index('donor').disease
lake_counts, lake_totals = {}, {}
for donor in lake_donors:
    for seg_name in ('S1', 'S2', 'S3'):
        rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.eq(seg_name)]
        lake_counts[(donor, seg_name)] = lake_sums.loc[rows].sum()
        lake_totals[(donor, seg_name)] = float(lake_meta.loc[rows, 'total'].sum())
lake_counts, lake_totals = pd.DataFrame(lake_counts), pd.Series(lake_totals)
healthy_lake = [d for d in lake_donors if lake_disease[d] == 'normal']
male_sn, female_sn = sn_sex.index[sn_sex.eq('male')], sn_sex.index[sn_sex.eq('female')]


def log2_cpm(counts_series, total):
    return np.log2(counts_series / total * 1e6 + 1)


def donor_contrasts(count_frame, totals, donors, contrast):
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    rows_contrast, rows_level = {}, {}
    for donor in donors:
        def level(parts):
            pooled = sum(count_frame[(donor, p)] for p in parts)
            return log2_cpm(pooled, sum(totals[(donor, p)] for p in parts))
        late_level = level([late])
        early_level = level(['S1', 'S2'] if early == 'early' else [early])
        rows_contrast[donor] = late_level - early_level
        rows_level[donor] = np.maximum(late_level, early_level)
    return pd.DataFrame(rows_contrast).T, pd.DataFrame(rows_level).T


def donor_gradient(count_frame, totals, donors, contrast):
    per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
    keep = level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()
    return paired_gradient(per_donor.loc[:, keep]), set(keep.index[keep.to_numpy()])


def reference_table(count_frame, totals, donors, contrast):
    per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
    per_donor = per_donor.loc[:, level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()]
    return moderated_contrast(per_donor, level.mean())


REF_GRADIENT = {(c, 'Lake'): donor_gradient(lake_counts, lake_totals, healthy_lake, 'S3-early' if c == 'S3c-early' else c)
                for c in CONTRASTS}
REF_GRADIENT.update({(c, 'snRNA male'): donor_gradient(sn_counts, sn_totals, male_sn, 'S3-early' if c == 'S3c-early' else c)
                     for c in CONTRASTS})
REF_TABLE = {(c, k): reference_table(*args, c) for c in CONTRASTS for k, args in
             (('mouse male', (sn_counts, sn_totals, male_sn)), ('mouse female', (sn_counts, sn_totals, female_sn)),
              ('human', (lake_counts, lake_totals, healthy_lake)))}
probe = pd.read_csv(probe_file).drop_duplicates('mouse_symbol').set_index('mouse_symbol')
balanced = set(probe.index[probe.probe_balance.eq('balanced')])
print(f'Lake healthy donors: {len(healthy_lake)}; mouse snRNA male/female donors: {len(male_sn)}/{len(female_sn)}')

## 3 · Notebook 43's primary index and symmetric classes, for any labels and evaluation genes

In [ ]:
def evaluate(segment, eval_genes, tag):
    """Primary conservation index and symmetric class counts (notebook 43) for given labels and genes."""
    eval_genes = set(eval_genes)
    cortical_s3 = np.zeros(len(specimen), bool)
    for name in expected['mouse']:
        limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
        cortical_s3 |= (specimen == name) & (segment == 'S3') & (depth <= limit)
    region = np.where((segment == 'S3') & np.isin(specimen, expected['mouse']) & ~cortical_s3, 'S3deep', segment)
    names_, sums = group_sums(counts, np.char.add(np.char.add(specimen.astype(str), '|'), region.astype(str)))
    frame = pd.DataFrame(sums, index=names_, columns=gene_space)
    parts = {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}
    pb, pb_library = {}, {}
    for name in names:
        for label, members in parts.items():
            rows = [f'{name}|{s}' for s in members if f'{name}|{s}' in frame.index]
            pb[(name, label)] = frame.loc[rows].sum()
            pb_library[(name, label)] = float(library[(specimen == name) & np.isin(region, members)].sum())
    pb = pd.DataFrame(pb)
    ours = pd.DataFrame({(contrast, name): log_ratio(pb[(name, late)], pb_library[(name, late)], pb[(name, early)], pb_library[(name, early)])
                         for contrast, (late, early) in CONTRASTS.items() for name in names}, index=gene_space)

    def count_filter(specimens, members):
        return pb[[(n, p) for n in specimens for p in members]].min(axis=1).ge(MIN_COUNTS)

    index_rows, count_rows, amplitude = [], [], {}
    for contrast, (late, early) in CONTRASTS.items():
        # Primary conservation index (notebook 43, section 5), restricted to evaluation genes.
        sets = {}
        for sp in expected:
            expressed = set(gene_space[count_filter(expected[sp], [late, early]).to_numpy()])
            sets[sp] = (paired_gradient(ours[[(contrast, n) for n in expected[sp]]].T.reset_index(drop=True)), expressed)
        lake_g, snm_g = REF_GRADIENT[(contrast, 'Lake')], REF_GRADIENT[(contrast, 'snRNA male')]
        genes = pd.Index(sorted(sets['human'][1] & sets['mouse'][1] & lake_g[1] & snm_g[1] & eval_genes))
        index_rows.append({'labels': tag, 'contrast': contrast,
                           **conservation_index(sets['human'][0], sets['mouse'][0], lake_g[0], snm_g[0], genes, n_boot=N_BOOT, seed=SEED)})
        # Symmetric classification (notebook 43, section 6): classify all genes, count evaluation genes.
        both = gene_space[(count_filter(expected['mouse'], [late, early]) & count_filter(expected['human'], [late, early])).to_numpy()]
        tabs = {}
        for sp in expected:
            replicates = ours.loc[both, [(contrast, n) for n in expected[sp]]].T.reset_index(drop=True)
            level = pd.concat([log2_cpm(pb[(n, p)], pb_library[(n, p)]) for n in expected[sp] for p in (late, early)], axis=1).mean(axis=1)
            tabs[sp] = moderated_contrast(replicates, level)
        amp_genes = [g for g in tabs['mouse'].index if g in eval_genes]
        reps = {sp: ours.loc[amp_genes, [(contrast, n) for n in expected[sp]]].to_numpy() for sp in expected}
        signal = {sp: r.mean(axis=1).var(ddof=1) - r.var(axis=1, ddof=1).mean() / r.shape[1] for sp, r in reps.items()}
        amplitude[contrast] = float(np.sqrt(max(signal['human'], 0) / signal['mouse']))
        mouse_calls, _ = zonation_calls(tabs['mouse'], alpha=ALPHA, effect=EFFECT, margin=MARGIN)
        mouse_calls = confirm(mouse_calls, REF_TABLE[(contrast, 'mouse male')], alpha=CONFIRM_ALPHA)
        human_calls, _ = zonation_calls(tabs['human'], alpha=ALPHA, effect=EFFECT * amplitude[contrast], margin=MARGIN * amplitude[contrast])
        human_calls = confirm(human_calls, REF_TABLE[(contrast, 'human')], alpha=CONFIRM_ALPHA)
        classes = pd.Series([cross_species_class(a, b) for a, b in zip(mouse_calls, human_calls)], index=mouse_calls.index)
        classes = confirm_flat(classes, mouse_calls, human_calls, REF_TABLE[(contrast, 'mouse male')], REF_TABLE[(contrast, 'human')],
                               alpha=CONFIRM_ALPHA, effect=EFFECT)
        counted = classes[classes.index.isin(balanced) & classes.index.isin(eval_genes)].value_counts()
        count_rows.append({'labels': tag, 'contrast': contrast, 'amplitude human/mouse': amplitude[contrast],
                           **{k: int(counted.get(k, 0)) for k in ('conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither')}})
        count_rows[-1]['mouse-only ÷ human-only'] = count_rows[-1]['mouse-only'] / max(count_rows[-1]['human-only'], 1)
    return pd.DataFrame(index_rows), pd.DataFrame(count_rows), amplitude


# Guard, every run: reviewed labels and all genes reproduce notebook 43 exactly.
check_index, check_counts, check_amplitude = evaluate(REVIEWED, gene_space, 'reviewed, all genes')
saved_index = pd.read_csv(reference43 / 'conservation_index.csv')
saved_index = saved_index[saved_index.version.str.startswith('primary')].set_index('contrast')
saved_counts = pd.read_csv(reference43 / 'class_counts_by_setting_and_contrast.csv')
saved_counts = saved_counts[saved_counts.setting.eq('symmetric (A3b + A8)')].set_index('contrast')
for row in check_index.itertuples():
    for field in ('index', 'ci_low', 'ci_high', 'genes'):
        if not np.isclose(getattr(row, field), saved_index.loc[row.contrast, field], rtol=0, atol=1e-9):
            raise ValueError(f'Index does not reproduce notebook 43: {row.contrast} {field}')
for _, row in check_counts.iterrows():
    for field in ('conserved', 'reversal', 'mouse-only', 'human-only'):
        if row[field] != saved_counts.loc[row['contrast'], field]:
            raise ValueError(f'Class counts do not reproduce notebook 43: {row["contrast"]} {field}')
print('Reproduces notebook 43 (reviewed labels, all genes): index and symmetric class counts identical.')
display(check_index.round(3))

## 4 · Label transfer on one gene half (notebook 42's centred score)

For each structure: its log-normalised profile centred on its specimen's PT mean, correlated with
the reference profile centred over S1/S2/S3; the label is the best match. Genes: measured in both
species, detected in ≥ 10% of the species' structures, expressed (max log2 ≥ 1) in the reference
and differing by ≥ 1 log2 between some pair of reference segments, and in the given half only.

In [ ]:
X_all = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)
genes_all = pd.Index(adata.var_names)
measured_all = adata.var.measured_in_both_inputs.to_numpy(bool)
detected = {sp: np.asarray((X_all[species == sp] > 0).mean(axis=0)).ravel() for sp in expected}
lake_means = pd.read_csv(results_root / 'pt_literature_deep' / 'lake_cortex_segment_means.csv', index_col='gene')
lake_means = lake_means.rename(columns=lambda c: c.replace('lake_log2cpm_', '')).reindex(genes_all)[['S1', 'S2', 'S3']]
micro_table = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
micro_means = bulk_segment_profiles(micro_table.drop_duplicates('Gene_symbol').set_index('Gene_symbol'), '_PT{seg}_').reindex(genes_all)[['S1', 'S2', 'S3']]
half_all = pd.Series(sha256_folds(genes_all, 2), index=genes_all)


def transfer_centred(sp, reference, half):
    diffs = pd.concat([(reference[a] - reference[b]).abs() for a, b in (('S2', 'S1'), ('S3', 'S2'), ('S3', 'S1'))], axis=1).max(axis=1)
    keep = (measured_all & (detected[sp] >= MIN_DETECTION) & reference.notna().all(axis=1).to_numpy()
            & (reference.max(axis=1) >= MIN_REFERENCE_LEVEL).to_numpy() & (diffs >= MIN_REFERENCE_DIFFERENCE).to_numpy()
            & (half_all.to_numpy() == half))
    cols = np.flatnonzero(keep)
    rows = np.flatnonzero(species == sp)
    dense = X_all[rows][:, cols].toarray()
    for name in np.unique(specimen[rows]):
        m = specimen[rows] == name
        dense[m] -= dense[m].mean(axis=0)
    dense /= np.linalg.norm(dense, axis=1, keepdims=True).clip(1e-12)
    ref = reference.iloc[cols].to_numpy(float)
    ref = ref - ref.mean(axis=1, keepdims=True)
    ref /= np.linalg.norm(ref, axis=0, keepdims=True).clip(1e-12)
    return rows, np.array(['S1', 'S2', 'S3'])[(dense @ ref).argmax(axis=1)], len(cols)


label_sets, transfer_rows = {}, []
for half in (0, 1):
    h_rows, h_labels, h_genes = transfer_centred('human', lake_means, half)
    m_rows, m_labels, m_genes = transfer_centred('mouse', micro_means, half)
    for setting in ('(i) human Lake, mouse reviewed', '(ii) human Lake, mouse microdissection', '(iii) reviewed'):
        seg = REVIEWED.copy()
        if not setting.startswith('(iii)'):
            seg[h_rows] = h_labels
        if setting.startswith('(ii)'):
            seg[m_rows] = m_labels
        label_sets[(setting, half)] = seg
        stem = {'(i)': 'crossfit_i', '(ii)': 'crossfit_ii', '(iii)': 'crossfit_iii'}[setting.split(' ')[0]] + f'_labels_from_half{half}'
        pd.DataFrame({'structure': adata.obs_names, 'segment': seg}).to_csv(label_dir / f'{stem}.csv', index=False)
        for sp in expected:
            m = species == sp
            transfer_rows.append({'setting': setting, 'labels from half': half, 'species': sp,
                                  'transfer genes': h_genes if sp == 'human' else m_genes,
                                  'agreement with reviewed': float((seg[m] == REVIEWED[m]).mean()),
                                  **{f'{s} fraction': float((seg[m] == s).mean()) for s in ('S1', 'S2', 'S3')}})
TRANSFERS = pd.DataFrame(transfer_rows)
TRANSFERS.to_csv(tables / 'crossfit_label_sets.csv', index=False)
display(TRANSFERS.round(3))

## 5 · Cross-fitted evaluation and the pre-registered decision

In [ ]:
index_parts, count_parts = [], []
for (setting, half), seg in label_sets.items():
    eval_half = 1 - half
    eval_genes = gene_space[HALF.to_numpy() == eval_half]
    idx, cnt, _ = evaluate(seg, eval_genes, setting)
    index_parts.append(idx.assign(**{'labels from half': half, 'evaluated on half': eval_half}))
    count_parts.append(cnt.assign(**{'labels from half': half, 'evaluated on half': eval_half}))
INDEX = pd.concat(index_parts, ignore_index=True)
COUNTS = pd.concat(count_parts, ignore_index=True)
INDEX.to_csv(tables / 'crossfit_conservation_index.csv', index=False)
COUNTS.to_csv(tables / 'crossfit_class_counts.csv', index=False)
check_index.to_csv(tables / 'reproduction_reviewed_all_genes_index.csv', index=False)
check_counts.to_csv(tables / 'reproduction_reviewed_all_genes_counts.csv', index=False)
display(INDEX[['labels', 'labels from half', 'contrast', 'index', 'ci_low', 'ci_high', 'cross_corrected', 'ceiling_human',
               'ceiling_mouse', 'genes']].round(3))
display(COUNTS[['labels', 'labels from half', 'contrast', 'amplitude human/mouse', 'conserved', 'reversal', 'mouse-only',
                'human-only', 'mouse-only ÷ human-only']].round(2))

primary = INDEX[INDEX.contrast.isin(['S2-S1', 'S3-early'])]
passes = (primary['index'] <= INDEX_MAX) & (primary.ci_high < INDEX_UPPER_MAX)
CLAIM_STANDS = bool(passes.all())
failing = primary.loc[~passes, ['labels', 'labels from half', 'contrast', 'index', 'ci_high']]
skew = COUNTS[COUNTS.contrast.isin(['S2-S1', 'S3c-early'])]
UNSKEWED = bool((skew['mouse-only ÷ human-only'] < SKEW_RATIO).all())
decision = pd.Series({'claim "largely species-specific" stands': CLAIM_STANDS,
                      'failing settings': '; '.join(f'{r["labels"]}, labels from half {r["labels from half"]}, {r["contrast"]} '
                                                    f'(index {r["index"]:.2f}, upper {r["ci_high"]:.2f})'
                                                    for _, r in failing.iterrows()) or 'none',
                      'species-only counts unskewed (ratio < 2, S2−S1 and S3c−early)': UNSKEWED,
                      'max index (S2−S1, S3−early)': float(primary['index'].max()),
                      'max upper bound (S2−S1, S3−early)': float(primary.ci_high.max())})
decision.to_csv(tables / 'decisions.csv')
display(decision.to_frame('value'))
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'decisions': decision.astype(str).to_dict(),
            'reproduction': 'notebook 43 reviewed index and symmetric counts reproduced exactly',
            'halves': HALF.value_counts().to_dict()}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))